
# BPE From Scratch + Legal Tokenizer Shootout

**Day 1 — AI Foundations · Practical 2 of 6 · Companion to the "Tokenization" deck**

> **Running in Google Colab:** the default **CPU runtime** is fine — no GPU needed. Run the
> setup cell below first, then proceed top to bottom.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Implement the **Byte Pair Encoding (BPE)** merge algorithm from scratch and trace it
   step by step on real legal-clause text
2. Explain why legal jargon (Latin phrases, defined terms) is often **more expensive** to
   tokenize than everyday English
3. Compare token counts for legal terms across a **general-purpose tokenizer (GPT-2)** and a
   **legal-domain tokenizer (LegalBERT)**
4. Connect the token-count differences directly to **context window and API cost** implications
   for a law firm building LLM-powered tools

## Notebook Workflow

```mermaid
flowchart TD
    A["Toy corpus of\nlegal clause snippets"] --> B["Part A:\nHand-build BPE merges"]
    B --> C["Watch vocabulary grow\ncharacter by character"]

    D["Legal jargon list\n(force majeure, estoppel, ...)"] --> E["Part B:\nTokenizer Shootout"]
    E --> F["GPT-2 tokenizer\n(general-purpose BPE)"]
    E --> G["LegalBERT tokenizer\n(legal-domain WordPiece)"]
    F --> H["Compare token counts"]
    G --> H
    H --> I["Cost / context-window\ntakeaways for a law firm"]
```

---


In [ ]:

# Install dependencies (used later in Part B for the tokenizer shootout).
# Running in Google Colab: this cell installs everything needed -- just run it.
%pip install -q transformers



## Part A — Byte Pair Encoding From Scratch

### Section 1 — Setup and Toy Corpus

Instead of the classic "low / lower / newest / widest" toy example from a generic NLP course,
we build our BPE vocabulary from real **contract clause language** so the merges we watch form
are ones that actually matter for legal text (e.g. "indemnif-" appearing across "indemnify,"
"indemnification," and "indemnitee").

Our toy corpus below is a small set of **word frequencies** drawn from the kind of recurring
vocabulary you'd see across CUAD/LEDGAR-style contract clauses (indemnification, termination,
governing-law provisions). Each word ends with a special end-of-word marker `</w>` so the
algorithm can tell "er" at the end of a word apart from "er" in the middle of one.


In [ ]:

from collections import Counter

# A tiny toy corpus: word -> frequency, built from recurring contract-clause vocabulary.
# In a real training run this would be counted across an entire corpus (e.g. Pile of Law's
# atticus_contracts subset); here we hand-pick frequencies to make the merges easy to follow.
corpus = {
    "indemnify": 5,
    "indemnification": 4,
    "indemnitee": 3,
    "indemnitor": 2,
    "terminate": 6,
    "termination": 5,
    "terminated": 3,
}

# Represent each word as a sequence of characters, space-separated, with an end-of-word marker
# e.g. "indemnify" -> "i n d e m n i f y </w>"
vocab = {
    " ".join(list(word)) + " </w>": freq
    for word, freq in corpus.items()
}

print("Starting vocabulary (character-level):\n")
for word, freq in vocab.items():
    print(f"  {word!r:60s} freq={freq}")



### Section 2 — The BPE Merge Loop

This is a direct implementation of the algorithm from the Tokenization deck:

1. Count every adjacent symbol pair across the corpus (weighted by word frequency)
2. Find the single most frequent pair
3. Merge that pair everywhere it occurs, creating one new symbol
4. Repeat

We wrap steps 1–3 in two small helper functions, then drive them in a loop that prints what
happened at each step so you can watch the vocabulary evolve in real time.


In [ ]:

def get_pair_counts(vocab):
    # Count frequency of every adjacent symbol pair across the vocabulary.
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_pair(pair, vocab):
    # Merge every occurrence of `pair` into a single new symbol across the vocabulary.
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab



Now let's run the loop and print each merge as it happens.


In [ ]:

NUM_MERGES = 10
merges = []
current_vocab = dict(vocab)

print(f"{'Step':<6}{'Merged Pair':<20}{'New Symbol':<20}\n" + "-" * 46)

for step in range(1, NUM_MERGES + 1):
    pairs = get_pair_counts(current_vocab)
    if not pairs:
        print("No more pairs to merge.")
        break

    best_pair = max(pairs, key=pairs.get)
    current_vocab = merge_pair(best_pair, current_vocab)
    merges.append(best_pair)

    new_symbol = "".join(best_pair)
    print(f"{step:<6}{str(best_pair):<20}{new_symbol:<20}")

print("\nFinal vocabulary after", NUM_MERGES, "merges:\n")
for word, freq in current_vocab.items():
    print(f"  {word!r:60s} freq={freq}")



### Section 3 — What Just Happened?

Watch the merge list: because "indemnif" appears inside **indemnify**, **indemnification**,
**indemnitee**, and **indemnitor**, BPE should discover that shared prefix as a standalone
subword symbol within the first several merges — purely from frequency counting, with no
hand-coded legal knowledge at all. Similarly for the shared "termin" root across "terminate,"
"termination," and "terminated."

This is the exact mechanism used to build real tokenizer vocabularies (GPT-2, RoBERTa, and
many others use byte-level BPE) — just run for tens of thousands of merges over a corpus of
billions of words instead of 10 merges over 7 words.



---

## Part B — Legal Tokenizer Shootout

### Section 4 — Setup

Now we compare how two **real, pretrained** tokenizers handle actual legal jargon:

- **GPT-2's tokenizer** — general-purpose byte-level BPE, trained on broad internet text
  (WebText), with no special attention to legal vocabulary
- **LegalBERT's tokenizer** (`nlpaueb/legal-bert-base-uncased`) — a WordPiece vocabulary
  trained specifically on ~12GB of legal text (legislation, court cases, contracts)

If domain-specific training data changes the vocabulary, we should see legal terms split into
**fewer, more meaningful pieces** by LegalBERT than by GPT-2.


In [ ]:

from transformers import AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
legalbert_tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

print("Both tokenizers loaded.")
print(f"  GPT-2 vocab size:      {gpt2_tokenizer.vocab_size:,}")
print(f"  LegalBERT vocab size:  {legalbert_tokenizer.vocab_size:,}")



### Section 5 — Head-to-Head Comparison

We'll run a list of common legal/Latin terms through both tokenizers and print how each one
gets split, plus the resulting token count. More tokens for the same word = more of your
context window and API budget consumed just to represent that one term.


In [ ]:

legal_terms = [
    "force majeure",
    "res judicata",
    "estoppel",
    "indemnification",
    "notwithstanding",
    "hereinafter",
    "arbitration",
    "jurisdiction",
]

print(f"{'Term':<20}{'GPT-2 tokens':<15}{'GPT-2 split':<45}{'LegalBERT tokens':<18}{'LegalBERT split'}")
print("-" * 140)

for term in legal_terms:
    gpt2_ids = gpt2_tokenizer.tokenize(term)
    legalbert_ids = legalbert_tokenizer.tokenize(term)

    print(
        f"{term:<20}"
        f"{len(gpt2_ids):<15}"
        f"{str(gpt2_ids):<45}"
        f"{len(legalbert_ids):<18}"
        f"{str(legalbert_ids)}"
    )



### Section 6 — Aggregate Comparison on a Full Clause

Individual words are illustrative, but what actually matters in production is the **total
token count for a realistic clause** — since that's what drives context-window usage and
per-request API cost. Let's tokenize a full indemnification clause with both tokenizers and
compare totals.


In [ ]:

clause = (
    "The Indemnitor shall, notwithstanding any provision hereinafter set forth, indemnify "
    "and hold harmless the Indemnitee from any claim arising out of force majeure events, "
    "unless barred by res judicata or subject to mandatory arbitration under the governing "
    "jurisdiction."
)

gpt2_tokens = gpt2_tokenizer.tokenize(clause)
legalbert_tokens = legalbert_tokenizer.tokenize(clause)

word_count = len(clause.split())

print(f"Clause word count (whitespace-split): {word_count}")
print(f"GPT-2 token count:                    {len(gpt2_tokens)}  ({len(gpt2_tokens)/word_count:.2f} tokens/word)")
print(f"LegalBERT token count:                {len(legalbert_tokens)}  ({len(legalbert_tokens)/word_count:.2f} tokens/word)")

print("\nGPT-2 tokens:")
print(gpt2_tokens)
print("\nLegalBERT tokens:")
print(legalbert_tokens)



## Section 7 — Try It Yourself

Paste a real clause from a document your firm works with (redact anything confidential) and
re-run the comparison. Watch especially for **defined terms** (capitalized terms like
"Confidential Information" or "Effective Date") and **Latin phrases** — these are the terms
most likely to fragment badly in a general-purpose tokenizer.


In [ ]:

your_clause = "The Effective Date of this Agreement shall be nunc pro tunc to the date first written above."

your_gpt2 = gpt2_tokenizer.tokenize(your_clause)
your_legalbert = legalbert_tokenizer.tokenize(your_clause)

print("GPT-2:     ", your_gpt2, f"({len(your_gpt2)} tokens)")
print("LegalBERT: ", your_legalbert, f"({len(your_legalbert)} tokens)")



## Key Takeaways

1. **BPE is purely statistical** — it discovers shared roots like "indemnif-" and "termin-"
   from frequency counts alone, no hand-coded legal knowledge required.
2. **Domain-specific training data changes the vocabulary.** A tokenizer trained on legal text
   (LegalBERT) tends to represent legal jargon more efficiently than a general-purpose one.
3. **Token count is not the same as word count** — and the gap widens for specialized
   vocabulary. This directly affects:
   - **Context window usage** — more tokens per clause means fewer clauses fit in a single
     LLM call
   - **API cost** — most providers bill per token, so inefficient tokenization is a real,
     measurable line-item cost at scale
4. When evaluating an LLM/embedding provider for legal-document workflows, checking how it
   tokenizes your firm's actual defined terms and Latin phrases is a legitimate due-diligence
   step, not a nice-to-have.

**Next up:** the *Finetuning & KV Cache* notebooks — adapting a model to contract-style
language, and making long-document generation fast.
